In [1]:
!pip install flask joblib requests scikit-learn numpy pandas

In [2]:
import numpy as np
import pandas as pd
import joblib
import requests
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

In [3]:
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42
)

# 2. Train the Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# 3. Evaluate on the test set
y_pred = rf_model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=iris.target_names))

# 4. Save the trained model
joblib.dump(rf_model, 'model.joblib')

# 5. Load back and verify predictions are identical
loaded_model = joblib.load('model.joblib')
loaded_pred = loaded_model.predict(X_test)

if np.array_equal(y_pred, loaded_pred):
    print("Verification Passed: Loaded model predictions match the original model perfectly.")
else:
    print("Verification Failed: Predictions differ.")

# 6. Save the target names
joblib.dump(iris.target_names, 'target_names.joblib')
print("Saved target_names.joblib successfully.")

Accuracy: 1.0

Classification Report:
               precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      1.00      1.00         9
   virginica       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30

Verification Passed: Loaded model predictions match the original model perfectly.
Saved target_names.joblib successfully.


In [5]:
import requests
import json

BASE_URL = "http://localhost:5000"

# 1. Health Check
print("--- Health Check ---")
response = requests.get(f"{BASE_URL}/health")
print(f"Status Code: {response.status_code}, Response: {response.json()}\n")

# 2. Single Prediction
print("--- Single Prediction ---")
valid_payload = {"features": [5.1, 3.5, 1.4, 0.2]}
response = requests.post(f"{BASE_URL}/predict", json=valid_payload)
print(f"Status Code: {response.status_code}, Response: {json.dumps(response.json(), indent=2)}\n")

# 3. Error Handling
print("--- Error Handling Tests ---")

error_payloads = [
    ({"wrong_key": [1, 2, 3, 4]}, "Missing 'features' key"),
    ({"features": [5.1, 3.5, 1.4]}, "Wrong number of features (3 instead of 4)"),
    ({"features": [5.1, 3.5, "a", 0.2]}, "Non-numeric values")
]

for payload, description in error_payloads:
    print(f"Testing: {description}")
    res = requests.post(f"{BASE_URL}/predict", json=payload)
    print(f"Status Code: {res.status_code}, Error: {res.json()}\n")

# 4. Batch Prediction
print("--- Batch Prediction ---")
# Using the first 5 samples from X_test we created in Task 1
batch_payload = {
    "samples": X_test[:5].tolist() 
}
response = requests.post(f"{BASE_URL}/predict_batch", json=batch_payload)
print(f"Status Code: {response.status_code}")
print("Response:", json.dumps(response.json(), indent=2))

--- Health Check ---
Status Code: 200, Response: {'status': 'healthy'}

--- Single Prediction ---
Status Code: 200, Response: {
  "predicted_class": "setosa",
  "probabilities": {
    "setosa": 1.0,
    "versicolor": 0.0,
    "virginica": 0.0
  }
}

--- Error Handling Tests ---
Testing: Missing 'features' key
Status Code: 400, Error: {'error': "Missing 'features' key in JSON body."}

Testing: Wrong number of features (3 instead of 4)
Status Code: 400, Error: {'error': "The 'features' list must contain exactly 4 values."}

Testing: Non-numeric values
Status Code: 400, Error: {'error': 'All feature values must be numeric.'}

--- Batch Prediction ---
Status Code: 200
Response: {
  "predictions": [
    {
      "predicted_class": "versicolor",
      "probabilities": {
        "setosa": 0.0,
        "versicolor": 0.99,
        "virginica": 0.01
      }
    },
    {
      "predicted_class": "setosa",
      "probabilities": {
        "setosa": 1.0,
        "versicolor": 0.0,
        "virginica

**Production Deployment Changes:** Running `app.run(debug=True)` is meant strictly for local development. For production, the Flask app should be served using a production-grade WSGI server like Gunicorn or uWSGI. This application should then be containerized using Docker to ensure environment consistency. A reverse proxy like Nginx would sit in front of the WSGI server to handle HTTPS termination, static file serving, and load balancing. Finally, all hardcoded variables and configurations should be moved to environment variables.
*   **Model Versioning:** If I train a new model, I would not just overwrite `model.joblib`. I would use a model registry (like MLflow or Weights & Biases) to track versions, hyperparameters, and artifacts. To deploy safely, I would use a "Shadow Deployment" (where the new model receives traffic but its predictions aren't returned to the user, only logged to compare against the live model) or "Canary Testing" (routing 5% of traffic to the new model) to ensure the new model performs well in the real world before a full rollout.
*   **Monitoring:** Beyond standard API metrics (CPU usage, memory, 500/400 HTTP error rates), ML-specific monitoring is critical. I would monitor **Prediction Latency** to ensure the model isn't slowing down the user experience. I would track **Input Drift** (e.g., if suddenly the API is receiving petal lengths of 500cm instead of 5cm) and **Prediction Drift** (e.g., if the model suddenly predicts 'setosa' 95% of the time, which could indicate a data pipeline bug). 
*   **Scaling for 1,000 Requests/Second:** A single Flask/Gunicorn instance would bottleneck quickly at 1,000 RPS. The architecture would need to scale horizontally. I would deploy multiple replicas of the Docker container managed by an orchestration tool like Kubernetes. A cloud Load Balancer would distribute incoming traffic across these pods. If the requests were coming in giant batches rather than individual HTTP requests, I would decouple the architecture: an API gateway drops the payload into a message queue (like RabbitMQ or Kafka), and worker nodes pull from the queue, batch the data, run predictions, and write to a database asynchronously.